# 🚀 Renest × ComfyUI One-Click Restore & Launch

**Deterministic Python dependency isolation & asset restore on Google Colab.**

### Step 1: Get Your 1-Click Colab Restore Code ⚡
Click any verified environment below to get your restore code (auto-copied to clipboard):
- 👉 **[app.renest.ai/colab](https://app.renest.ai/colab)** (Browse all environments)
- 🔗 **[FLUX.2 Klein 4B (Text to Image)](https://app.renest.ai/colab?env=klein)**
- 🔗 **[Z-Image-Turbo (Text to Image)](https://app.renest.ai/colab?env=z-image)**
- 🔗 **[Wan 2.2 TI2V 5B (Text to Video)](https://app.renest.ai/colab?env=wan)**
- 🔗 **[Wan Animate 2 (Video Animate)](https://app.renest.ai/colab?env=animate)**

*(Or use any private Hand-off code from a collaborator)*

### Step 2: Paste & Click Play Once ▶️
Paste your code into `handoff_code` below and run Cell 1. It restores everything, verifies a test render on your GPU, and automatically opens your public ComfyUI WebUI link.

In [ ]:
# @title 1. Restore & Launch ComfyUI WebUI 🚀
# @markdown Paste your Hand-off / Restore code below and run this cell:
handoff_code = "PASTE_YOUR_RESTORE_CODE_HERE"  # @param {type:"string"}
workspace = "/content/ComfyUI"  # @param {type:"string"}

WORKSPACE = workspace

import json
import os
import re
import shutil
import subprocess
import sys

# 1. Hardware Pre-flight Check & Transparent Disclosure
print("🔍 Checking Google Colab GPU environment...")
try:
    import torch
    has_cuda = torch.cuda.is_available()
except ImportError:
    has_cuda = False

if not has_cuda:
    print("\n" + "=" * 70)
    print("❌ NO GPU DETECTED!")
    print("   This ComfyUI environment requires an NVIDIA GPU to run.")
    print("   Google Colab is currently connected to a standard CPU instance.")
    print("")
    print("   👉 To enable GPU in 5 seconds:")
    print("   1. Go to top menu: Runtime -> Change runtime type")
    print("   2. Hardware accelerator: select 'T4 GPU' (free)")
    print("   3. Click 'Save', then click Run on this cell again.")
    print("=" * 70 + "\n")
    raise SystemExit("Stopped: GPU accelerator not connected. Please switch to T4 GPU in Runtime settings.")

gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
total_disk, used_disk, free_disk = shutil.disk_usage("/content")
print(f"✅ GPU: {gpu_name} ({vram_gib:.1f} GiB VRAM)")
print(f"✅ Disk Space: {free_disk / (1024**3):.1f} GiB free on /content\n")

# 2. Parse input code / command
code = handoff_code.strip()
if not code or code == "PASTE_YOUR_RESTORE_CODE_HERE":
    print("\n" + "=" * 70)
    print("❌ MISSING RESTORE CODE!")
    print("   Get your 1-click restore code here:")
    print("   👉 https://app.renest.ai/colab")
    print("")
    print("   1. Click 'Get Colab Code' on your preferred environment")
    print("   2. Paste the copied code into `handoff_code` above and run again.")
    print("=" * 70 + "\n")
    raise SystemExit("Stopped: Please provide a restore code in the handoff_code field.")

grant_file = "/content/grant.json"
grant_arg = None

# Case A: User pasted the entire shell script from web console (contains embedded JSON)
json_match = re.search(r"(\{[\s\S]*\})", code)
if json_match:
    try:
        parsed = json.loads(json_match.group(1))
        if "grant_id" in parsed:
            with open(grant_file, "w", encoding="utf-8") as gf:
                json.dump(parsed, gf, indent=2)
            grant_arg = grant_file
    except Exception:
        pass

# Case B: Standard input parsing (file path, raw JSON, CLI command, or bare ULID token)
if grant_arg is None:
    if "--grant" in code:
        m = re.search(r"--grant\s+([^\s]+)", code)
        if m and m.group(1).strip("\"'") != "grant.json":
            code = m.group(1).strip("\"'")

    if os.path.isfile(code):
        grant_arg = code
    elif code.startswith("{"):
        with open(grant_file, "w", encoding="utf-8") as gf:
            gf.write(code)
        grant_arg = grant_file
    else:
        token_match = re.search(r"[0-9A-HJ-NP-Za-km-z]{26}", code)
        clean_id = token_match.group(0) if token_match else code.strip()
        envelope = {
            "grant_version": "2",
            "grant_id": clean_id,
            "origin": "https://api.renest.ai"
        }
        with open(grant_file, "w", encoding="utf-8") as gf:
            json.dump(envelope, gf, indent=2)
        grant_arg = grant_file


# Pre-provision WebUI launcher and download cloudflared in parallel while restoring
helper_script = "/tmp/renest_colab.py"
with open(helper_script, "w", encoding="utf-8") as hf:
    hf.write('#!/usr/bin/env python3\n"""Renest Colab Runner: Launches ComfyUI and creates a public WebUI tunnel on Google Colab.\n\nRules & Architecture:\n1. Environment must be pre-restored via `renest restore`.\n2. Serves as a host supervisor calling `renest start` and running cloudflared tunnel.\n3. Does not modify renest CLI or server internals. Pure standard library implementation.\n4. User-facing text in English per project constitution.\n5. Clean lifecycle: guarantees termination of both child processes on any exit path.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import deque\nimport os\nimport platform\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nfrom typing import Optional\n\n\nCLOUDFLARED_URLS = {\n    ("Linux", "x86_64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",\n    ("Linux", "aarch64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64",\n    ("Darwin", "x86_64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-amd64",\n    ("Darwin", "arm64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-arm64",\n}\n\n\ndef ensure_cloudflared(dest_dir: str = "/tmp") -> str:\n    """Ensure cloudflared binary is available, downloading standalone binary if missing."""\n    system_bin = shutil.which("cloudflared")\n    if system_bin:\n        return system_bin\n\n    local_bin = os.path.join(dest_dir, "cloudflared")\n    if os.path.isfile(local_bin) and os.access(local_bin, os.X_OK) and os.path.getsize(local_bin) > 1_000_000:\n        return local_bin\n\n    os_type = platform.system()\n    machine = platform.machine()\n    download_url = CLOUDFLARED_URLS.get((os_type, machine))\n    if not download_url:\n        download_url = CLOUDFLARED_URLS[("Linux", "x86_64")]\n\n    print(f"[Renest Colab] Downloading Cloudflare tunnel binary ({os_type}/{machine})...", flush=True)\n    tmp_bin = local_bin + ".tmp"\n    try:\n        urllib.request.urlretrieve(download_url, tmp_bin)\n        os.chmod(tmp_bin, 0o755)\n        os.replace(tmp_bin, local_bin)\n    except Exception as e:\n        raise RuntimeError(f"Failed to download cloudflared: {e}. Check network connection.") from e\n\n    return local_bin\n\n\ndef extract_tunnel_url(log_line: str) -> Optional[str]:\n    """Extract trycloudflare.com URL from cloudflared log line."""\n    match = re.search(r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com", log_line)\n    if match:\n        return match.group(0)\n    return None\n\n\ndef wait_for_http_ready(port: int, timeout: float = 120.0) -> bool:\n    """Poll local HTTP port until ComfyUI answers HTTP 200, or timeout expires."""\n    deadline = time.monotonic() + timeout\n    endpoints = [f"http://127.0.0.1:{port}/system_stats", f"http://127.0.0.1:{port}/"]\n    while time.monotonic() < deadline:\n        for url in endpoints:\n            try:\n                req = urllib.request.Request(url, headers={"User-Agent": "Renest-Colab-HealthCheck/1.0"})\n                with urllib.request.urlopen(req, timeout=2.0) as resp:\n                    if resp.status == 200:\n                        return True\n            except (urllib.error.URLError, OSError):\n                pass\n        time.sleep(1.0)\n    return False\n\n\ndef wait_for_public_url(url: str, timeout: float = 20.0) -> tuple[bool, str]:\n    """Poll a public tunnel URL until DNS + edge routing return HTTP 200 end-to-end."""\n    deadline = time.monotonic() + timeout\n    probe_url = f"{url.rstrip(\'/\')}/system_stats"\n    last_err = "timed out waiting for HTTP 200"\n    while time.monotonic() < deadline:\n        try:\n            req = urllib.request.Request(probe_url, headers={"User-Agent": "Renest-Colab-HealthCheck/1.0"})\n            with urllib.request.urlopen(req, timeout=4.0) as resp:\n                if 200 <= resp.status < 400:\n                    return True, ""\n                last_err = f"HTTP {resp.status}"\n        except urllib.error.HTTPError as e:\n            if e.code == 404:\n                # Non-ComfyUI root/fallback path may still answer 404 on /system_stats while / answers 200\n                try:\n                    req_root = urllib.request.Request(url.rstrip("/") + "/", headers={"User-Agent": "Renest-Colab-HealthCheck/1.0"})\n                    with urllib.request.urlopen(req_root, timeout=4.0) as resp2:\n                        if 200 <= resp2.status < 400:\n                            return True, ""\n                except Exception as e2:\n                    last_err = str(e2)\n            else:\n                last_err = f"HTTP {e.code}"\n        except (urllib.error.URLError, OSError) as e:\n            last_err = str(e)\n        time.sleep(1.0)\n    return False, last_err\n\n\ndef _terminate_proc(proc: Optional[subprocess.Popen], timeout: float = 5.0) -> None:\n    """Terminate a subprocess and its entire process group, ensuring zero orphans.\n\n    Reaps direct child processes with wait() to avoid zombie/defunct entries.\n    Raises RuntimeError if any process in the group remains alive after SIGKILL.\n    """\n    if proc is None:\n        return\n\n    pid = proc.pid\n    try:\n        pgid = os.getpgid(pid)\n    except (AttributeError, OSError, ProcessLookupError):\n        pgid = pid\n\n    def _group_alive() -> bool:\n        try:\n            os.killpg(pgid, 0)\n            return True\n        except (AttributeError, OSError, ProcessLookupError):\n            return proc.poll() is None\n\n    def _reap_proc() -> None:\n        try:\n            proc.wait(timeout=0.2)\n        except (subprocess.TimeoutExpired, OSError, AttributeError):\n            pass\n\n    if not _group_alive():\n        _reap_proc()\n        return\n\n    # 1. Try graceful SIGINT first to allow clean application teardown\n    try:\n        try:\n            os.killpg(pgid, signal.SIGINT)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.send_signal(signal.SIGINT)\n        deadline = time.monotonic() + 3.0\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.1)\n    except OSError:\n        pass\n\n    # 2. Fall back to SIGTERM\n    try:\n        try:\n            os.killpg(pgid, signal.SIGTERM)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.terminate()\n        deadline = time.monotonic() + timeout\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.1)\n    except OSError:\n        pass\n\n    # 3. Force SIGKILL on process group\n    try:\n        try:\n            os.killpg(pgid, signal.SIGKILL)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.kill()\n        deadline = time.monotonic() + 2.0\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.05)\n    except OSError:\n        pass\n\n    _reap_proc()\n    if _group_alive():\n        raise RuntimeError(\n            f"Process teardown failed: process group {pgid} (PID {pid}) still active after SIGKILL"\n        )\n\n\ndef _cleanup_procs(app_proc: Optional[subprocess.Popen], tunnel_proc: Optional[subprocess.Popen]) -> None:\n    """Ensure both child processes are terminated and waited for, preventing orphan processes.\n\n    Raises RuntimeError if teardown fails to terminate either process group.\n    """\n    errors = []\n    for name, p, t in [("tunnel", tunnel_proc, 3.0), ("app", app_proc, 5.0)]:\n        try:\n            _terminate_proc(p, timeout=t)\n        except Exception as e:\n            errors.append(f"{name}: {e}")\n    if errors:\n        msg = f"Process teardown failed: {\'; \'.join(errors)}"\n        print(f"[Renest Colab] Error: {msg}", file=sys.stderr, flush=True)\n        raise RuntimeError(msg)\n\n\ndef _dump_tail(label: str, tail: deque[str]) -> None:\n    if not tail:\n        return\n    print(f"--- Last {len(tail)} lines from {label} ---", file=sys.stderr, flush=True)\n    for line in tail:\n        print(f"  [{label}] {line}", file=sys.stderr, flush=True)\n\n\ndef _stop_tunnel(tunnel_proc: Optional[subprocess.Popen]) -> None:\n    """Terminate an optional tunnel process without failing the caller."""\n    if tunnel_proc is None:\n        return\n    try:\n        _terminate_proc(tunnel_proc, timeout=2.0)\n    except Exception as exc:\n        print(f"[Renest Colab] Warning during tunnel cleanup: {exc}", file=sys.stderr, flush=True)\n\n\ndef launch_webui(\n    workspace: str,\n    port: int = 8188,\n    listen: Optional[str] = None,\n    cloudflared_path: Optional[str] = None,\n    timeout_tunnel: float = 45.0,\n    timeout_http: float = 120.0,\n    allow_local_fallback: bool = False,\n) -> int:\n    """Launch `renest start` and optional cloudflared tunnel, draining pipes and guarding lifecycle."""\n    if not os.path.isdir(workspace):\n        print(f"Error: Workspace directory does not exist: {workspace}", file=sys.stderr, flush=True)\n        print("Please run `renest restore` first to restore the environment.", file=sys.stderr, flush=True)\n        return 1\n\n    cf_bin: Optional[str] = cloudflared_path\n    if cf_bin is None:\n        try:\n            cf_bin = ensure_cloudflared()\n        except RuntimeError as exc:\n            if not allow_local_fallback:\n                raise\n            print(f"[Renest Colab] Warning: Cloudflare binary unavailable ({exc}); continuing with local port {port}.", file=sys.stderr, flush=True)\n            cf_bin = None\n\n    app_proc: Optional[subprocess.Popen] = None\n    tunnel_proc: Optional[subprocess.Popen] = None\n\n    def _sig_handler(signum, frame):\n        raise KeyboardInterrupt()\n\n    prev_int = signal.signal(signal.SIGINT, _sig_handler)\n    prev_term = signal.signal(signal.SIGTERM, _sig_handler)\n\n    try:\n        print(f"[Renest Colab] Starting ComfyUI via `renest start` (port {port})...", flush=True)\n        start_cmd = [\n            "renest",\n            "start",\n            "--dir",\n            workspace,\n            "--port",\n            str(port),\n        ]\n        if listen:\n            start_cmd.extend(["--listen", listen])\n\n        app_proc = subprocess.Popen(\n            start_cmd,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            text=True,\n            bufsize=1,\n            start_new_session=True,\n        )\n\n        app_tail: deque[str] = deque(maxlen=40)\n        tunnel_tail: deque[str] = deque(maxlen=30)\n        tunnel_url_holder: list[str] = []\n        tunnel_found_event = threading.Event()\n\n        def drain_app():\n            if not app_proc or not app_proc.stdout:\n                return\n            for line in app_proc.stdout:\n                clean = line.rstrip()\n                if clean:\n                    app_tail.append(clean)\n                if any(\n                    k in line\n                    for k in (\n                        "[start]",\n                        "Starting server",\n                        "Import times",\n                        "To see the GUI",\n                        "Traceback",\n                        "Error",\n                        "FATAL",\n                    )\n                ):\n                    print(f"[ComfyUI] {clean}", flush=True)\n\n        t_app = threading.Thread(target=drain_app, daemon=True)\n        t_app.start()\n\n        t_tunnel: Optional[threading.Thread] = None\n        if cf_bin is not None:\n            tunnel_cmd = [\n                cf_bin,\n                "tunnel",\n                "--protocol",\n                "http2",\n                "--no-autoupdate",\n                "--url",\n                f"http://127.0.0.1:{port}",\n            ]\n            print("[Renest Colab] Establishing Cloudflare tunnel (HTTP/2)...", flush=True)\n            tunnel_proc = subprocess.Popen(\n                tunnel_cmd,\n                stdout=subprocess.PIPE,\n                stderr=subprocess.STDOUT,\n                text=True,\n                bufsize=1,\n                start_new_session=True,\n            )\n\n            def drain_tunnel():\n                if not tunnel_proc or not tunnel_proc.stdout:\n                    return\n                for line in tunnel_proc.stdout:\n                    clean = line.rstrip()\n                    if clean:\n                        tunnel_tail.append(clean)\n                    if not tunnel_found_event.is_set():\n                        url = extract_tunnel_url(line)\n                        if url:\n                            tunnel_url_holder.append(url)\n                            tunnel_found_event.set()\n\n            t_tunnel = threading.Thread(target=drain_tunnel, daemon=True)\n            t_tunnel.start()\n\n            # 1. Wait for tunnel URL, polling early-exit status of both children\n            tunnel_deadline = time.monotonic() + timeout_tunnel\n            while not tunnel_found_event.is_set():\n                if app_proc.poll() is not None:\n                    t_app.join(timeout=1.0)\n                    print(\n                        f"Error: renest start exited prematurely (code {app_proc.returncode}).",\n                        file=sys.stderr,\n                        flush=True,\n                    )\n                    _dump_tail("ComfyUI", app_tail)\n                    return app_proc.returncode or 1\n                if tunnel_proc.poll() is not None:\n                    t_tunnel.join(timeout=1.0)\n                    msg = f"cloudflared exited prematurely (code {tunnel_proc.returncode})."\n                    if allow_local_fallback:\n                        print(f"[Renest Colab] Warning: {msg} Continuing with local WebUI fallback.", file=sys.stderr, flush=True)\n                        _dump_tail("cloudflared", tunnel_tail)\n                        tunnel_proc = None\n                        break\n                    print(f"Error: {msg}", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    return tunnel_proc.returncode or 1\n                if time.monotonic() >= tunnel_deadline:\n                    msg = f"Timed out waiting for Cloudflare tunnel URL ({timeout_tunnel}s)."\n                    if allow_local_fallback:\n                        print(f"[Renest Colab] Warning: {msg} Continuing with local WebUI fallback.", file=sys.stderr, flush=True)\n                        _dump_tail("cloudflared", tunnel_tail)\n                        _stop_tunnel(tunnel_proc)\n                        tunnel_proc = None\n                        break\n                    print(f"Error: {msg}", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    _dump_tail("ComfyUI", app_tail)\n                    return 1\n                tunnel_found_event.wait(timeout=0.25)\n\n        tunnel_url: Optional[str] = tunnel_url_holder[0] if tunnel_url_holder else None\n        if tunnel_url:\n            print(f"[Renest Colab] Tunnel endpoint ready: {tunnel_url}", flush=True)\n\n        # 2. Health check ComfyUI HTTP endpoint: fail fast if app_proc exits prematurely\n        print(f"[Renest Colab] Waiting for ComfyUI to answer on port {port}...", flush=True)\n        http_deadline = time.monotonic() + timeout_http\n        http_ready = False\n        while True:\n            if app_proc.poll() is not None:\n                t_app.join(timeout=1.0)\n                print(\n                    f"Error: ComfyUI process terminated with exit code {app_proc.returncode}",\n                    file=sys.stderr,\n                    flush=True,\n                )\n                _dump_tail("ComfyUI", app_tail)\n                return app_proc.returncode or 1\n            remaining = http_deadline - time.monotonic()\n            if remaining <= 0:\n                break\n            slice_timeout = min(1.0, max(0.05, remaining))\n            if wait_for_http_ready(port=port, timeout=slice_timeout):\n                http_ready = True\n                break\n\n        if not http_ready:\n            print(f"Error: ComfyUI HTTP health check timed out after {timeout_http}s.", file=sys.stderr, flush=True)\n            _dump_tail("ComfyUI", app_tail)\n            if app_proc.poll() is not None:\n                print(\n                    f"Error: ComfyUI process terminated with exit code {app_proc.returncode}",\n                    file=sys.stderr,\n                    flush=True,\n                )\n                return app_proc.returncode or 1\n            return 1\n\n        # 3. Verify tunnel end-to-end before declaring public URL ready\n        if tunnel_url:\n            print(f"[Renest Colab] Verifying Cloudflare tunnel reachability ({tunnel_url})...", flush=True)\n            tunnel_ok, tunnel_err = wait_for_public_url(tunnel_url, timeout=20.0)\n            if not tunnel_ok:\n                msg = f"Cloudflare tunnel {tunnel_url} failed end-to-end check ({tunnel_err})."\n                _dump_tail("cloudflared", tunnel_tail)\n                if allow_local_fallback:\n                    print(f"[Renest Colab] Warning: {msg} Continuing with Colab Proxy.", file=sys.stderr, flush=True)\n                    _stop_tunnel(tunnel_proc)\n                    tunnel_proc = None\n                    tunnel_url = None\n                else:\n                    print(f"Error: {msg}", file=sys.stderr, flush=True)\n                    return 1\n\n        # 4. Declare ready (with verified Cloudflare tunnel URL or local Colab fallback marker)\n        if tunnel_url:\n            banner = [\n                "",\n                "═" * 70,\n                "  🎉 ComfyUI WebUI is ready!",\n                f"  👉 Public WebUI URL (verified HTTP 200): {tunnel_url}",\n                "     (If *.trycloudflare.com is blocked on your network, use the Colab Proxy link below.)",\n                "═" * 70,\n                "",\n            ]\n        else:\n            banner = [\n                "",\n                "═" * 70,\n                f"  🎉 ComfyUI WebUI is running locally on port {port}!",\n                f"  [Renest Colab] LOCAL_WEBUI_READY port={port}",\n                "  👉 Cloudflare tunnel was unavailable; use the Colab Proxy link below.",\n                "═" * 70,\n                "",\n            ]\n        print("\\n".join(banner), flush=True)\n\n        # 4. Supervise until interrupt or child process termination\n        while True:\n            if app_proc.poll() is not None:\n                t_app.join(timeout=1.0)\n                print(f"\\n[Renest Colab] ComfyUI process exited (code {app_proc.returncode}).", file=sys.stderr, flush=True)\n                _dump_tail("ComfyUI", app_tail)\n                return app_proc.returncode\n            if tunnel_proc is not None and tunnel_proc.poll() is not None:\n                if t_tunnel is not None:\n                    t_tunnel.join(timeout=1.0)\n                msg = f"cloudflared process exited (code {tunnel_proc.returncode})."\n                if allow_local_fallback:\n                    print(f"\\n[Renest Colab] Warning: {msg} Keeping ComfyUI alive on local port {port}.", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    print(f"[Renest Colab] LOCAL_WEBUI_READY port={port}", flush=True)\n                    tunnel_proc = None\n                else:\n                    print(f"\\n[Renest Colab] {msg}", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    return tunnel_proc.returncode\n            time.sleep(1)\n\n    except KeyboardInterrupt:\n        print("\\n[Renest Colab] Shutting down services...", flush=True)\n        return 0\n    finally:\n        signal.signal(signal.SIGINT, prev_int)\n        signal.signal(signal.SIGTERM, prev_term)\n        # Guarantee both processes are cleaned up on any exit path\n        _cleanup_procs(app_proc, tunnel_proc)\n\n\ndef main(argv: Optional[list[str]] = None) -> int:\n    parser = argparse.ArgumentParser(\n        description="Renest Colab Helper: Launches ComfyUI and establishes public tunnel."\n    )\n    subparsers = parser.add_subparsers(dest="subcommand", required=True)\n\n    launch_p = subparsers.add_parser("launch", help="Launch ComfyUI and tunnel")\n    launch_p.add_argument(\n        "--dir",\n        default="/content/ComfyUI",\n        help="Workspace directory (default: /content/ComfyUI)",\n    )\n    launch_p.add_argument(\n        "--port",\n        type=int,\n        default=8188,\n        help="ComfyUI listen port (default: 8188)",\n    )\n    launch_p.add_argument(\n        "--listen",\n        default=None,\n        help="Optional ComfyUI listen host override (only valid when nest records a listen flag)",\n    )\n    launch_p.add_argument(\n        "--timeout-tunnel",\n        type=float,\n        default=45.0,\n        help="Max seconds to wait for tunnel URL (default: 45.0)",\n    )\n    launch_p.add_argument(\n        "--timeout-http",\n        type=float,\n        default=120.0,\n        help="Max seconds to wait for ComfyUI HTTP readiness (default: 120.0)",\n    )\n    launch_p.add_argument(\n        "--allow-local-fallback",\n        action="store_true",\n        help="Keep ComfyUI running locally if cloudflared quick tunnel fails or times out",\n    )\n\n    args = parser.parse_args(argv)\n\n    try:\n        if args.subcommand == "launch":\n            return launch_webui(\n                workspace=args.dir,\n                port=args.port,\n                listen=args.listen,\n                timeout_tunnel=args.timeout_tunnel,\n                timeout_http=args.timeout_http,\n                allow_local_fallback=args.allow_local_fallback,\n            )\n    except RuntimeError as e:\n        print(f"Error: {e}", file=sys.stderr, flush=True)\n        return 1\n\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')

def _prefetch_cloudflared():
    import platform as _plat, shutil as _shutil, urllib.request as _req
    target = "/tmp/cloudflared"
    if _shutil.which("cloudflared") or (os.path.isfile(target) and os.access(target, os.X_OK)):
        return
    urls = {
        ("Linux", "x86_64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ("Linux", "aarch64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64",
    }
    url = urls.get((_plat.system(), _plat.machine()), urls[("Linux", "x86_64")])
    tmp = target + ".tmp"
    try:
        _req.urlretrieve(url, tmp)
        os.chmod(tmp, 0o755)
        os.replace(tmp, target)
    except Exception:
        pass

import threading as _threading
_cf_thread = _threading.Thread(target=_prefetch_cloudflared, daemon=True)
_cf_thread.start()

# 3. Environment isolation for Google Colab
# Suppress UV_SYSTEM_PYTHON so uv isolates packages inside .venv
os.environ["UV_SYSTEM_PYTHON"] = "0"
os.environ["UV_NO_CONFIG"] = "1"

print("📦 Installing / Updating Renest CLI & uv...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv", "renest"], check=True)

# Install standalone CPython 3.11 to ensure binary PyTorch CUDA wheel compatibility
print("🐍 Ensuring Python 3.11 runtime environment...")
subprocess.run(["uv", "python", "install", "3.11"], check=True)

# 4. Run 5-Gate Restore Pipeline with Real-Time Disclosure
import time

print("\n🔄 Starting Renest 5-Gate Restore Pipeline (live telemetry enabled):")
print("   [1/5] Doctor S0: Hardware compatibility & CUDA driver inspection")
print("   [2/5] S1 & S2: Ranged multi-stream asset download & SHA-256 verification")
print("   [3/5] S3: Deterministic Python dependency isolation with uv")
print("   [4/5] S4: ComfyUI process startup & runtime health check")
print("   [5/5] S5: Baseline test render verification (proves pixels generate on GPU)\n", flush=True)

restore_cmd = [
    "renest", "--json", "restore",
    "--grant", grant_arg,
    "--dir", WORKSPACE,
    "--yes",
    "--force",
]

proc = subprocess.Popen(
    restore_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

last_progress_at = 0.0
last_progress_pct = -1.0
final_result = None

assert proc.stdout is not None
for raw_line in proc.stdout:
    line = raw_line.strip()
    if not line:
        continue
    if not line.startswith("{"):
        print(line, flush=True)
        continue
    try:
        ev = json.loads(line)
    except Exception:
        print(line, flush=True)
        continue

    etype = ev.get("type")
    stage = ev.get("stage", "")
    if etype == "stage_start":
        desc = ev.get("desc", "")
        print(f"\n▶ [{stage}] {desc}...", flush=True)
        last_progress_at = 0.0
        last_progress_pct = -1.0
    elif etype == "progress":
        pct = float(ev.get("percent") or 0.0)
        now = time.monotonic()
        done_bytes = int(ev.get("bytes_done") or 0)
        total_bytes = int(ev.get("bytes_total") or 0)
        done_gib = done_bytes / (1024 ** 3)
        total_gib = total_bytes / (1024 ** 3)
        mbps = float(ev.get("speed_mbps") or 0.0)
        if done_bytes < 1024 * 1024 and pct < 100.0 and mbps < 1.0:
            continue
        if (now - last_progress_at >= 3.0) or (int(pct // 10) > int(last_progress_pct // 10)) or pct >= 100.0:
            last_progress_at = now
            last_progress_pct = pct
            eta = ""
            if mbps >= 1.0 and done_bytes >= 10 * 1024 * 1024 and total_gib > done_gib:
                secs = (total_gib - done_gib) * (1024 ** 3) * 8 / (mbps * 1e6)
                eta = (
                    f" · ~{int(secs // 60)}m {int(secs % 60):02d}s left"
                    if secs >= 60
                    else f" · ~{int(secs)}s left"
                )
            srcs = ev.get("active_sources") or []
            src_note = f" [{', '.join(str(s) for s in srcs[:2])}]" if srcs else ""
            print(f"  ⏳ [{stage}] {pct:5.1f}%  {done_gib:.2f}/{total_gib:.2f} GiB  ({mbps:.0f} Mbps{eta}){src_note}", flush=True)
    elif etype == "log":
        msg = ev.get("message", "")
        if msg:
            tag = f"[{stage}] " if stage else ""
            print(f"  {tag}{msg}", flush=True)
    elif etype == "stage_done":
        dur = float(ev.get("duration_s") or 0.0)
        detail = ev.get("detail") or ""
        suffix = f" — {detail}" if detail else ""
        print(f"✅ [{stage}] completed in {dur:.1f}s{suffix}", flush=True)
    elif etype == "error":
        ec = ev.get("error_class", "ERROR")
        human = ev.get("human", "")
        detail = ev.get("detail", "")
        print(f"\n❌ [{stage}/{ec}] {human}", flush=True)
        if detail:
            print(f"   {detail}", flush=True)
    elif etype == "result":
        final_result = ev

class _Res:
    returncode = proc.wait()

restore_res = _Res()

if restore_res.returncode != 0:
    code = restore_res.returncode
    error_hints = {
        63: "S0/CUDA_BLOCK: GPU or CUDA driver mismatch. Please verify your Colab runtime is set to T4 GPU (Runtime -> Change runtime type).",
        62: "S0/PYTHON_BLOCK: Host Python version incompatible with environment lockfile.",
        65: "S0/DISK_INSUFFICIENT: Free disk space is insufficient for this model pack.",
        13: "S1/CREDENTIAL_EXPIRED: The restore grant code has expired. Please claim a fresh starter or regenerate code.",
        11: "S1/NETWORK_INTERRUPTED: Download interrupted. Rerun this cell to resume from where it stopped.",
        23: "S2/HASH_MISMATCH: Downloaded file failed SHA-256 checksum verification.",
        31: "S3/TORCH_CUDA_CONFLICT: Torch and CUDA library conflict during environment build.",
    }
    hint = error_hints.get(code, f"Stage failure (code {code}). Review the log output above for details.")
    print("\n" + "=" * 70)
    print(f"❌ Restore halted with exit code {code}")
    print(f"💡 Diagnostic: {hint}")
    failure = (final_result or {}).get("failure") if isinstance(final_result, dict) else None
    if isinstance(failure, dict):
        if failure.get("human"):
            print(f"❗ Reason: {failure['human']}")
        if failure.get("detail"):
            print(f"📋 Detail:\n{failure['detail']}")
    print("=" * 70 + "\n")
    raise SystemExit(f"Restore stopped (exit code {code}): {hint}")

# Display verified test render image if produced
output_dirs = [
    os.path.join(WORKSPACE, "output"),
    os.path.join(WORKSPACE, "ComfyUI", "output"),
]
found_images = []
for od in output_dirs:
    if os.path.isdir(od):
        found_images.extend([
            os.path.join(od, f) for f in os.listdir(od)
            if f.endswith(".png") and not f.startswith(".")
        ])

if found_images:
    found_images.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    latest_img = found_images[0]
    print(f"\n🖼️ Verified out-of-the-box render: {os.path.basename(latest_img)} ({os.path.getsize(latest_img)} bytes)")
    try:
        from IPython.display import Image, display
        display(Image(latest_img, width=512))
    except Exception:
        pass

print("\n✅ All 5 gates passed! ComfyUI environment verified on this GPU.")
print("🚀 Launching ComfyUI WebUI & Cloudflare public link (keep this cell running while using WebUI)...", flush=True)
if "_cf_thread" in globals() and _cf_thread.is_alive():
    _cf_thread.join(timeout=15.0)
from collections import deque as _deque

def _show_colab_proxy(port=8188):
    try:
        from google.colab import output as _colab_output
        print("🔗 Colab Official Proxy:", flush=True)
        _colab_output.serve_kernel_port_as_window(port, anchor_text="🚀 Open ComfyUI WebUI (Colab Official Proxy)")
        _proxy_url = _colab_output.eval_js(f"google.colab.kernel.proxyPort({port})")
        print(f"   Direct proxy URL: {_proxy_url}", flush=True)
        print("   (Note: If Colab Proxy shows 403, allow third-party cookies for *.colab.dev or focus the address bar and press Enter.)", flush=True)
    except Exception:
        pass

_launch_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_tail = _deque(maxlen=40)
_proxy_shown = False
_webui_proc = subprocess.Popen(
    [sys.executable, helper_script, "launch", "--dir", WORKSPACE, "--allow-local-fallback"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=_launch_env,
)
if _webui_proc.stdout:
    for _line in _webui_proc.stdout:
        print(_line, end="", flush=True)
        _clean = _line.rstrip()
        if _clean:
            _tail.append(_clean)
        if not _proxy_shown and (
            "[Renest Colab] LOCAL_WEBUI_READY" in _line
            or "ComfyUI WebUI is ready!" in _line
        ):
            _proxy_shown = True
            _show_colab_proxy(8188)
_rc = _webui_proc.wait()
if _rc != 0:
    _tail_txt = "\n".join(_tail)
    raise RuntimeError(
        f"ComfyUI WebUI runner exited with code {_rc}.\nLast output lines:\n{_tail_txt}"
    )

In [ ]:
# @title 2. (Optional) Re-launch ComfyUI WebUI & Public Tunnel 🎨
# @markdown Starts ComfyUI and creates a public web access link via Cloudflare tunnel.

import os
import subprocess
import sys

# 1. Verify Step 1 completed successfully before launching
WORKSPACE = globals().get("WORKSPACE", "/content/ComfyUI")
start_facts = os.path.join(WORKSPACE, ".renest", "start.json")
venv_py = os.path.join(WORKSPACE, ".venv", "bin", "python")

if not (os.path.isdir(WORKSPACE) and os.path.isfile(start_facts) and os.path.isfile(venv_py)):
    raise RuntimeError(
        f"❌ Workspace '{WORKSPACE}' does not contain a verified restored environment.\n"
        "Please run Step 1 (Restore) first and ensure it finishes with exit code 0."
    )

# 2. Ensure supervisor runner is present (auto-provisioned if running on fresh Colab)
helper_script = "/tmp/renest_colab.py"
with open(helper_script, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n"""Renest Colab Runner: Launches ComfyUI and creates a public WebUI tunnel on Google Colab.\n\nRules & Architecture:\n1. Environment must be pre-restored via `renest restore`.\n2. Serves as a host supervisor calling `renest start` and running cloudflared tunnel.\n3. Does not modify renest CLI or server internals. Pure standard library implementation.\n4. User-facing text in English per project constitution.\n5. Clean lifecycle: guarantees termination of both child processes on any exit path.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import deque\nimport os\nimport platform\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nfrom typing import Optional\n\n\nCLOUDFLARED_URLS = {\n    ("Linux", "x86_64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",\n    ("Linux", "aarch64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64",\n    ("Darwin", "x86_64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-amd64",\n    ("Darwin", "arm64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-arm64",\n}\n\n\ndef ensure_cloudflared(dest_dir: str = "/tmp") -> str:\n    """Ensure cloudflared binary is available, downloading standalone binary if missing."""\n    system_bin = shutil.which("cloudflared")\n    if system_bin:\n        return system_bin\n\n    local_bin = os.path.join(dest_dir, "cloudflared")\n    if os.path.isfile(local_bin) and os.access(local_bin, os.X_OK) and os.path.getsize(local_bin) > 1_000_000:\n        return local_bin\n\n    os_type = platform.system()\n    machine = platform.machine()\n    download_url = CLOUDFLARED_URLS.get((os_type, machine))\n    if not download_url:\n        download_url = CLOUDFLARED_URLS[("Linux", "x86_64")]\n\n    print(f"[Renest Colab] Downloading Cloudflare tunnel binary ({os_type}/{machine})...", flush=True)\n    tmp_bin = local_bin + ".tmp"\n    try:\n        urllib.request.urlretrieve(download_url, tmp_bin)\n        os.chmod(tmp_bin, 0o755)\n        os.replace(tmp_bin, local_bin)\n    except Exception as e:\n        raise RuntimeError(f"Failed to download cloudflared: {e}. Check network connection.") from e\n\n    return local_bin\n\n\ndef extract_tunnel_url(log_line: str) -> Optional[str]:\n    """Extract trycloudflare.com URL from cloudflared log line."""\n    match = re.search(r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com", log_line)\n    if match:\n        return match.group(0)\n    return None\n\n\ndef wait_for_http_ready(port: int, timeout: float = 120.0) -> bool:\n    """Poll local HTTP port until ComfyUI answers HTTP 200, or timeout expires."""\n    deadline = time.monotonic() + timeout\n    endpoints = [f"http://127.0.0.1:{port}/system_stats", f"http://127.0.0.1:{port}/"]\n    while time.monotonic() < deadline:\n        for url in endpoints:\n            try:\n                req = urllib.request.Request(url, headers={"User-Agent": "Renest-Colab-HealthCheck/1.0"})\n                with urllib.request.urlopen(req, timeout=2.0) as resp:\n                    if resp.status == 200:\n                        return True\n            except (urllib.error.URLError, OSError):\n                pass\n        time.sleep(1.0)\n    return False\n\n\ndef wait_for_public_url(url: str, timeout: float = 20.0) -> tuple[bool, str]:\n    """Poll a public tunnel URL until DNS + edge routing return HTTP 200 end-to-end."""\n    deadline = time.monotonic() + timeout\n    probe_url = f"{url.rstrip(\'/\')}/system_stats"\n    last_err = "timed out waiting for HTTP 200"\n    while time.monotonic() < deadline:\n        try:\n            req = urllib.request.Request(probe_url, headers={"User-Agent": "Renest-Colab-HealthCheck/1.0"})\n            with urllib.request.urlopen(req, timeout=4.0) as resp:\n                if 200 <= resp.status < 400:\n                    return True, ""\n                last_err = f"HTTP {resp.status}"\n        except urllib.error.HTTPError as e:\n            if e.code == 404:\n                # Non-ComfyUI root/fallback path may still answer 404 on /system_stats while / answers 200\n                try:\n                    req_root = urllib.request.Request(url.rstrip("/") + "/", headers={"User-Agent": "Renest-Colab-HealthCheck/1.0"})\n                    with urllib.request.urlopen(req_root, timeout=4.0) as resp2:\n                        if 200 <= resp2.status < 400:\n                            return True, ""\n                except Exception as e2:\n                    last_err = str(e2)\n            else:\n                last_err = f"HTTP {e.code}"\n        except (urllib.error.URLError, OSError) as e:\n            last_err = str(e)\n        time.sleep(1.0)\n    return False, last_err\n\n\ndef _terminate_proc(proc: Optional[subprocess.Popen], timeout: float = 5.0) -> None:\n    """Terminate a subprocess and its entire process group, ensuring zero orphans.\n\n    Reaps direct child processes with wait() to avoid zombie/defunct entries.\n    Raises RuntimeError if any process in the group remains alive after SIGKILL.\n    """\n    if proc is None:\n        return\n\n    pid = proc.pid\n    try:\n        pgid = os.getpgid(pid)\n    except (AttributeError, OSError, ProcessLookupError):\n        pgid = pid\n\n    def _group_alive() -> bool:\n        try:\n            os.killpg(pgid, 0)\n            return True\n        except (AttributeError, OSError, ProcessLookupError):\n            return proc.poll() is None\n\n    def _reap_proc() -> None:\n        try:\n            proc.wait(timeout=0.2)\n        except (subprocess.TimeoutExpired, OSError, AttributeError):\n            pass\n\n    if not _group_alive():\n        _reap_proc()\n        return\n\n    # 1. Try graceful SIGINT first to allow clean application teardown\n    try:\n        try:\n            os.killpg(pgid, signal.SIGINT)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.send_signal(signal.SIGINT)\n        deadline = time.monotonic() + 3.0\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.1)\n    except OSError:\n        pass\n\n    # 2. Fall back to SIGTERM\n    try:\n        try:\n            os.killpg(pgid, signal.SIGTERM)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.terminate()\n        deadline = time.monotonic() + timeout\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.1)\n    except OSError:\n        pass\n\n    # 3. Force SIGKILL on process group\n    try:\n        try:\n            os.killpg(pgid, signal.SIGKILL)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.kill()\n        deadline = time.monotonic() + 2.0\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.05)\n    except OSError:\n        pass\n\n    _reap_proc()\n    if _group_alive():\n        raise RuntimeError(\n            f"Process teardown failed: process group {pgid} (PID {pid}) still active after SIGKILL"\n        )\n\n\ndef _cleanup_procs(app_proc: Optional[subprocess.Popen], tunnel_proc: Optional[subprocess.Popen]) -> None:\n    """Ensure both child processes are terminated and waited for, preventing orphan processes.\n\n    Raises RuntimeError if teardown fails to terminate either process group.\n    """\n    errors = []\n    for name, p, t in [("tunnel", tunnel_proc, 3.0), ("app", app_proc, 5.0)]:\n        try:\n            _terminate_proc(p, timeout=t)\n        except Exception as e:\n            errors.append(f"{name}: {e}")\n    if errors:\n        msg = f"Process teardown failed: {\'; \'.join(errors)}"\n        print(f"[Renest Colab] Error: {msg}", file=sys.stderr, flush=True)\n        raise RuntimeError(msg)\n\n\ndef _dump_tail(label: str, tail: deque[str]) -> None:\n    if not tail:\n        return\n    print(f"--- Last {len(tail)} lines from {label} ---", file=sys.stderr, flush=True)\n    for line in tail:\n        print(f"  [{label}] {line}", file=sys.stderr, flush=True)\n\n\ndef _stop_tunnel(tunnel_proc: Optional[subprocess.Popen]) -> None:\n    """Terminate an optional tunnel process without failing the caller."""\n    if tunnel_proc is None:\n        return\n    try:\n        _terminate_proc(tunnel_proc, timeout=2.0)\n    except Exception as exc:\n        print(f"[Renest Colab] Warning during tunnel cleanup: {exc}", file=sys.stderr, flush=True)\n\n\ndef launch_webui(\n    workspace: str,\n    port: int = 8188,\n    listen: Optional[str] = None,\n    cloudflared_path: Optional[str] = None,\n    timeout_tunnel: float = 45.0,\n    timeout_http: float = 120.0,\n    allow_local_fallback: bool = False,\n) -> int:\n    """Launch `renest start` and optional cloudflared tunnel, draining pipes and guarding lifecycle."""\n    if not os.path.isdir(workspace):\n        print(f"Error: Workspace directory does not exist: {workspace}", file=sys.stderr, flush=True)\n        print("Please run `renest restore` first to restore the environment.", file=sys.stderr, flush=True)\n        return 1\n\n    cf_bin: Optional[str] = cloudflared_path\n    if cf_bin is None:\n        try:\n            cf_bin = ensure_cloudflared()\n        except RuntimeError as exc:\n            if not allow_local_fallback:\n                raise\n            print(f"[Renest Colab] Warning: Cloudflare binary unavailable ({exc}); continuing with local port {port}.", file=sys.stderr, flush=True)\n            cf_bin = None\n\n    app_proc: Optional[subprocess.Popen] = None\n    tunnel_proc: Optional[subprocess.Popen] = None\n\n    def _sig_handler(signum, frame):\n        raise KeyboardInterrupt()\n\n    prev_int = signal.signal(signal.SIGINT, _sig_handler)\n    prev_term = signal.signal(signal.SIGTERM, _sig_handler)\n\n    try:\n        print(f"[Renest Colab] Starting ComfyUI via `renest start` (port {port})...", flush=True)\n        start_cmd = [\n            "renest",\n            "start",\n            "--dir",\n            workspace,\n            "--port",\n            str(port),\n        ]\n        if listen:\n            start_cmd.extend(["--listen", listen])\n\n        app_proc = subprocess.Popen(\n            start_cmd,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            text=True,\n            bufsize=1,\n            start_new_session=True,\n        )\n\n        app_tail: deque[str] = deque(maxlen=40)\n        tunnel_tail: deque[str] = deque(maxlen=30)\n        tunnel_url_holder: list[str] = []\n        tunnel_found_event = threading.Event()\n\n        def drain_app():\n            if not app_proc or not app_proc.stdout:\n                return\n            for line in app_proc.stdout:\n                clean = line.rstrip()\n                if clean:\n                    app_tail.append(clean)\n                if any(\n                    k in line\n                    for k in (\n                        "[start]",\n                        "Starting server",\n                        "Import times",\n                        "To see the GUI",\n                        "Traceback",\n                        "Error",\n                        "FATAL",\n                    )\n                ):\n                    print(f"[ComfyUI] {clean}", flush=True)\n\n        t_app = threading.Thread(target=drain_app, daemon=True)\n        t_app.start()\n\n        t_tunnel: Optional[threading.Thread] = None\n        if cf_bin is not None:\n            tunnel_cmd = [\n                cf_bin,\n                "tunnel",\n                "--protocol",\n                "http2",\n                "--no-autoupdate",\n                "--url",\n                f"http://127.0.0.1:{port}",\n            ]\n            print("[Renest Colab] Establishing Cloudflare tunnel (HTTP/2)...", flush=True)\n            tunnel_proc = subprocess.Popen(\n                tunnel_cmd,\n                stdout=subprocess.PIPE,\n                stderr=subprocess.STDOUT,\n                text=True,\n                bufsize=1,\n                start_new_session=True,\n            )\n\n            def drain_tunnel():\n                if not tunnel_proc or not tunnel_proc.stdout:\n                    return\n                for line in tunnel_proc.stdout:\n                    clean = line.rstrip()\n                    if clean:\n                        tunnel_tail.append(clean)\n                    if not tunnel_found_event.is_set():\n                        url = extract_tunnel_url(line)\n                        if url:\n                            tunnel_url_holder.append(url)\n                            tunnel_found_event.set()\n\n            t_tunnel = threading.Thread(target=drain_tunnel, daemon=True)\n            t_tunnel.start()\n\n            # 1. Wait for tunnel URL, polling early-exit status of both children\n            tunnel_deadline = time.monotonic() + timeout_tunnel\n            while not tunnel_found_event.is_set():\n                if app_proc.poll() is not None:\n                    t_app.join(timeout=1.0)\n                    print(\n                        f"Error: renest start exited prematurely (code {app_proc.returncode}).",\n                        file=sys.stderr,\n                        flush=True,\n                    )\n                    _dump_tail("ComfyUI", app_tail)\n                    return app_proc.returncode or 1\n                if tunnel_proc.poll() is not None:\n                    t_tunnel.join(timeout=1.0)\n                    msg = f"cloudflared exited prematurely (code {tunnel_proc.returncode})."\n                    if allow_local_fallback:\n                        print(f"[Renest Colab] Warning: {msg} Continuing with local WebUI fallback.", file=sys.stderr, flush=True)\n                        _dump_tail("cloudflared", tunnel_tail)\n                        tunnel_proc = None\n                        break\n                    print(f"Error: {msg}", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    return tunnel_proc.returncode or 1\n                if time.monotonic() >= tunnel_deadline:\n                    msg = f"Timed out waiting for Cloudflare tunnel URL ({timeout_tunnel}s)."\n                    if allow_local_fallback:\n                        print(f"[Renest Colab] Warning: {msg} Continuing with local WebUI fallback.", file=sys.stderr, flush=True)\n                        _dump_tail("cloudflared", tunnel_tail)\n                        _stop_tunnel(tunnel_proc)\n                        tunnel_proc = None\n                        break\n                    print(f"Error: {msg}", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    _dump_tail("ComfyUI", app_tail)\n                    return 1\n                tunnel_found_event.wait(timeout=0.25)\n\n        tunnel_url: Optional[str] = tunnel_url_holder[0] if tunnel_url_holder else None\n        if tunnel_url:\n            print(f"[Renest Colab] Tunnel endpoint ready: {tunnel_url}", flush=True)\n\n        # 2. Health check ComfyUI HTTP endpoint: fail fast if app_proc exits prematurely\n        print(f"[Renest Colab] Waiting for ComfyUI to answer on port {port}...", flush=True)\n        http_deadline = time.monotonic() + timeout_http\n        http_ready = False\n        while True:\n            if app_proc.poll() is not None:\n                t_app.join(timeout=1.0)\n                print(\n                    f"Error: ComfyUI process terminated with exit code {app_proc.returncode}",\n                    file=sys.stderr,\n                    flush=True,\n                )\n                _dump_tail("ComfyUI", app_tail)\n                return app_proc.returncode or 1\n            remaining = http_deadline - time.monotonic()\n            if remaining <= 0:\n                break\n            slice_timeout = min(1.0, max(0.05, remaining))\n            if wait_for_http_ready(port=port, timeout=slice_timeout):\n                http_ready = True\n                break\n\n        if not http_ready:\n            print(f"Error: ComfyUI HTTP health check timed out after {timeout_http}s.", file=sys.stderr, flush=True)\n            _dump_tail("ComfyUI", app_tail)\n            if app_proc.poll() is not None:\n                print(\n                    f"Error: ComfyUI process terminated with exit code {app_proc.returncode}",\n                    file=sys.stderr,\n                    flush=True,\n                )\n                return app_proc.returncode or 1\n            return 1\n\n        # 3. Verify tunnel end-to-end before declaring public URL ready\n        if tunnel_url:\n            print(f"[Renest Colab] Verifying Cloudflare tunnel reachability ({tunnel_url})...", flush=True)\n            tunnel_ok, tunnel_err = wait_for_public_url(tunnel_url, timeout=20.0)\n            if not tunnel_ok:\n                msg = f"Cloudflare tunnel {tunnel_url} failed end-to-end check ({tunnel_err})."\n                _dump_tail("cloudflared", tunnel_tail)\n                if allow_local_fallback:\n                    print(f"[Renest Colab] Warning: {msg} Continuing with Colab Proxy.", file=sys.stderr, flush=True)\n                    _stop_tunnel(tunnel_proc)\n                    tunnel_proc = None\n                    tunnel_url = None\n                else:\n                    print(f"Error: {msg}", file=sys.stderr, flush=True)\n                    return 1\n\n        # 4. Declare ready (with verified Cloudflare tunnel URL or local Colab fallback marker)\n        if tunnel_url:\n            banner = [\n                "",\n                "═" * 70,\n                "  🎉 ComfyUI WebUI is ready!",\n                f"  👉 Public WebUI URL (verified HTTP 200): {tunnel_url}",\n                "     (If *.trycloudflare.com is blocked on your network, use the Colab Proxy link below.)",\n                "═" * 70,\n                "",\n            ]\n        else:\n            banner = [\n                "",\n                "═" * 70,\n                f"  🎉 ComfyUI WebUI is running locally on port {port}!",\n                f"  [Renest Colab] LOCAL_WEBUI_READY port={port}",\n                "  👉 Cloudflare tunnel was unavailable; use the Colab Proxy link below.",\n                "═" * 70,\n                "",\n            ]\n        print("\\n".join(banner), flush=True)\n\n        # 4. Supervise until interrupt or child process termination\n        while True:\n            if app_proc.poll() is not None:\n                t_app.join(timeout=1.0)\n                print(f"\\n[Renest Colab] ComfyUI process exited (code {app_proc.returncode}).", file=sys.stderr, flush=True)\n                _dump_tail("ComfyUI", app_tail)\n                return app_proc.returncode\n            if tunnel_proc is not None and tunnel_proc.poll() is not None:\n                if t_tunnel is not None:\n                    t_tunnel.join(timeout=1.0)\n                msg = f"cloudflared process exited (code {tunnel_proc.returncode})."\n                if allow_local_fallback:\n                    print(f"\\n[Renest Colab] Warning: {msg} Keeping ComfyUI alive on local port {port}.", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    print(f"[Renest Colab] LOCAL_WEBUI_READY port={port}", flush=True)\n                    tunnel_proc = None\n                else:\n                    print(f"\\n[Renest Colab] {msg}", file=sys.stderr, flush=True)\n                    _dump_tail("cloudflared", tunnel_tail)\n                    return tunnel_proc.returncode\n            time.sleep(1)\n\n    except KeyboardInterrupt:\n        print("\\n[Renest Colab] Shutting down services...", flush=True)\n        return 0\n    finally:\n        signal.signal(signal.SIGINT, prev_int)\n        signal.signal(signal.SIGTERM, prev_term)\n        # Guarantee both processes are cleaned up on any exit path\n        _cleanup_procs(app_proc, tunnel_proc)\n\n\ndef main(argv: Optional[list[str]] = None) -> int:\n    parser = argparse.ArgumentParser(\n        description="Renest Colab Helper: Launches ComfyUI and establishes public tunnel."\n    )\n    subparsers = parser.add_subparsers(dest="subcommand", required=True)\n\n    launch_p = subparsers.add_parser("launch", help="Launch ComfyUI and tunnel")\n    launch_p.add_argument(\n        "--dir",\n        default="/content/ComfyUI",\n        help="Workspace directory (default: /content/ComfyUI)",\n    )\n    launch_p.add_argument(\n        "--port",\n        type=int,\n        default=8188,\n        help="ComfyUI listen port (default: 8188)",\n    )\n    launch_p.add_argument(\n        "--listen",\n        default=None,\n        help="Optional ComfyUI listen host override (only valid when nest records a listen flag)",\n    )\n    launch_p.add_argument(\n        "--timeout-tunnel",\n        type=float,\n        default=45.0,\n        help="Max seconds to wait for tunnel URL (default: 45.0)",\n    )\n    launch_p.add_argument(\n        "--timeout-http",\n        type=float,\n        default=120.0,\n        help="Max seconds to wait for ComfyUI HTTP readiness (default: 120.0)",\n    )\n    launch_p.add_argument(\n        "--allow-local-fallback",\n        action="store_true",\n        help="Keep ComfyUI running locally if cloudflared quick tunnel fails or times out",\n    )\n\n    args = parser.parse_args(argv)\n\n    try:\n        if args.subcommand == "launch":\n            return launch_webui(\n                workspace=args.dir,\n                port=args.port,\n                listen=args.listen,\n                timeout_tunnel=args.timeout_tunnel,\n                timeout_http=args.timeout_http,\n                allow_local_fallback=args.allow_local_fallback,\n            )\n    except RuntimeError as e:\n        print(f"Error: {e}", file=sys.stderr, flush=True)\n        return 1\n\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')

print("🚀 Launching ComfyUI WebUI supervisor...")
from collections import deque as _deque

def _show_colab_proxy(port=8188):
    try:
        from google.colab import output as _colab_output
        print("🔗 Colab Official Proxy:", flush=True)
        _colab_output.serve_kernel_port_as_window(port, anchor_text="🚀 Open ComfyUI WebUI (Colab Official Proxy)")
        _proxy_url = _colab_output.eval_js(f"google.colab.kernel.proxyPort({port})")
        print(f"   Direct proxy URL: {_proxy_url}", flush=True)
        print("   (Note: If Colab Proxy shows 403, allow third-party cookies for *.colab.dev or focus the address bar and press Enter.)", flush=True)
    except Exception:
        pass

_launch_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_tail = _deque(maxlen=40)
_proxy_shown = False
_webui_proc = subprocess.Popen(
    [sys.executable, helper_script, "launch", "--dir", WORKSPACE, "--allow-local-fallback"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=_launch_env,
)
if _webui_proc.stdout:
    for _line in _webui_proc.stdout:
        print(_line, end="", flush=True)
        _clean = _line.rstrip()
        if _clean:
            _tail.append(_clean)
        if not _proxy_shown and (
            "[Renest Colab] LOCAL_WEBUI_READY" in _line
            or "ComfyUI WebUI is ready!" in _line
        ):
            _proxy_shown = True
            _show_colab_proxy(8188)
_rc = _webui_proc.wait()
if _rc != 0:
    _tail_txt = "\n".join(_tail)
    raise RuntimeError(
        f"ComfyUI WebUI runner exited with code {_rc}.\nLast output lines:\n{_tail_txt}"
    )

---
### 💡 Next Steps: Save your modified workflows and models
Once you finish generating images or installing new custom nodes, run `renest pack` to store verified working dependencies and assets into your own nest.


In [ ]:
# @title 3. (Optional) Pack Environment into a New Nest 💾
!renest pack --dir "$WORKSPACE"
